<a href="https://colab.research.google.com/github/VelizarZ/crypto_price_prediction/blob/main/cursor_thinking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Probabilistic Tracker — Synth Competition

Implements all four research-backed layers from the plan: Student-t fat tails (VAR paper), GARCH(1,1) volatility (VAR paper), NGBoost with CRPScore (NGBoost paper), and quantile calibration (Koenker & Hallock).

In [ ]:
# Install dependencies
# ngboost ships with crunch_synth in Colab; add it explicitly here for safety
%pip install crunch_synth ngboost -q


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import os
from datetime import datetime, timezone, timedelta
from scipy import stats as sp_stats
from scipy.optimize import minimize
from tqdm.auto import tqdm

from crunch_synth import (
    TrackerBase,
    TrackerEvaluator,
    FORECAST_PROFILES,
    SUPPORTED_ASSETS,
    pricedb,
    load_test_prices_once,
    load_initial_price_histories_once,
    visualize_price_data,
    count_evaluations,
    plot_quarantine,
    plot_scores,
)


# Advanced Probabilistic Tracker — Synth Competition

Four improvements over the baseline Gaussian tracker, each grounded in one of the research papers:

| Layer | Paper | What it adds |
|-------|-------|--------------|
| **1 — Fat-tailed distributions** | VAR + fat tails + SV (Chiu et al. 2015) | Replaces Normal with Student-t; df estimated from kurtosis so heavy-tailed assets (BTC, SOL) get wider tails automatically |
| **2 — GARCH(1,1) volatility** | Same VAR paper | Replaces naïve EWMA with a proper GARCH(1,1) conditional variance forecast; captures volatility clustering and mean-reversion |
| **3 — NGBoost w/ CRPScore** | NGBoost (Duan et al. 2020, arXiv:1910.03225) | Gradient-boosted model that **directly optimises CRPS** (the competition metric) to predict (μ, σ) from 20+ engineered features |
| **4 — Quantile calibration** | Quantile Regression (Koenker & Hallock 2001) | Fits empirical 5th–95th percentile spread vs theoretical Student-t spread to produce a per-asset scale correction |


In [ ]:
# ============================================================================
# Advanced Probabilistic Tracker
# ============================================================================

class AdvancedProbabilisticTracker(TrackerBase):
    """
    Probabilistic tracker combining four research-backed layers:

    Layer 1 (VAR paper)      – Student-t distributions for fat tails
    Layer 2 (VAR paper)      – GARCH(1,1) conditional variance forecasting
    Layer 3 (NGBoost paper)  – NGBoost w/ CRPScore for CRPS-optimised learning
    Layer 4 (QR paper)       – Quantile calibration scale correction
    """

    # ── constants ──────────────────────────────────────────────────────────
    _BASE_RES      = 5 * 60   # 5-minute bars (seconds)
    _RETRAIN_EVERY = 50       # NGBoost retrain cadence (# predict calls)
    _HISTORY_DAYS  = 7        # price history window
    _NGBOOST_TREES = 200      # number of NGBoost estimators
    _EWMA_LAMBDA   = 0.94     # RiskMetrics decay

    def __init__(self):
        super().__init__()
        # Per-asset state caches
        self._garch   = {}    # (omega, alpha, beta, sigma2_next)
        self._df      = {}    # Student-t degrees of freedom
        self._scale   = {}    # quantile calibration scale
        self._ngb     = {}    # NGBoost models
        self._calls   = {}    # prediction counter
        self._fcols   = None  # feature column order (set on first training)

    # =========================================================================
    # Layer 1 – Student-t degrees-of-freedom estimation
    # =========================================================================

    def _estimate_df(self, returns: np.ndarray) -> float:
        """
        Method-of-moments: for Student-t, excess_kurtosis = 6 / (df - 4).
        Therefore df = 6 / excess_kurtosis + 4.  Clipped to [3, 30].
        df < 3  → undefined variance; df > 30 ≈ Gaussian.
        """
        if len(returns) < 20:
            return 5.0
        r = returns - np.mean(returns)
        m2 = np.mean(r ** 2)
        m4 = np.mean(r ** 4)
        if m2 < 1e-14:
            return 5.0
        excess_kurt = m4 / m2 ** 2 - 3.0
        df = (6.0 / excess_kurt + 4.0) if excess_kurt > 0 else 30.0
        return float(np.clip(df, 3.0, 30.0))

    # =========================================================================
    # Layer 2 – GARCH(1,1) fitting and multi-step forecasting
    # =========================================================================

    def _fit_garch(self, returns: np.ndarray):
        """
        Gaussian MLE for GARCH(1,1):
            sigma2_t = omega + alpha * eps_{t-1}^2 + beta * sigma2_{t-1}

        Returns (omega, alpha, beta, sigma2_T+1) where sigma2_T+1 is the
        one-step-ahead conditional variance from the end of the series.
        """
        n = len(returns)
        var0 = float(np.var(returns))
        fallback = (var0 * 0.05, 0.1, 0.85, var0)

        if n < 30 or var0 < 1e-14:
            return fallback

        def _variance_path(params):
            omega, alpha, beta = params
            s2 = np.empty(n)
            s2[0] = var0
            for t in range(1, n):
                s2[t] = omega + alpha * returns[t - 1] ** 2 + beta * s2[t - 1]
            return s2

        def _nll(params):
            omega, alpha, beta = params
            if omega <= 0 or alpha < 0 or beta < 0 or alpha + beta >= 0.9999:
                return 1e10
            s2 = _variance_path(params)
            if np.any(s2 <= 0):
                return 1e10
            return float(0.5 * np.sum(np.log(s2) + returns ** 2 / s2))

        try:
            res = minimize(
                _nll,
                x0=[var0 * 0.05, 0.1, 0.85],
                method="L-BFGS-B",
                bounds=[(1e-9, None), (1e-6, 0.4), (1e-6, 0.9999)],
                options={"maxiter": 200},
            )
            if res.success and np.all(np.isfinite(res.x)):
                omega, alpha, beta = res.x
                s2 = _variance_path(res.x)
                sigma2_next = float(
                    max(omega + alpha * returns[-1] ** 2 + beta * s2[-1], 1e-14)
                )
                return (omega, alpha, beta, sigma2_next)
        except Exception:
            pass

        return fallback

    def _garch_forecast(
        self, omega: float, alpha: float, beta: float,
        sigma2: float, h: int
    ) -> float:
        """
        h-step-ahead GARCH conditional variance (analytical):
            E[sigma2_{t+h}] = long_run + persistence^h * (sigma2_{t+1} - long_run)
        """
        persistence = alpha + beta
        if persistence >= 1.0:
            return sigma2
        long_run = omega / (1.0 - persistence)
        return float(max(long_run + persistence ** h * (sigma2 - long_run), 1e-14))

    # =========================================================================
    # Layer 3 – NGBoost feature engineering + model
    # =========================================================================

    def _features(
        self,
        prices: np.ndarray,
        returns: np.ndarray,
        times: np.ndarray,
    ) -> dict:
        """Build a feature dict from the current price/return series."""
        n = len(returns)
        p = float(prices[-1])
        feat = {}

        # Rolling return means & realised vols at 5 horizons
        for w, lbl in [(6, "30m"), (12, "1h"), (24, "2h"), (48, "4h"), (288, "24h")]:
            w = min(w, n)
            r_win = returns[-w:]
            feat[f"mean_{lbl}"] = float(np.mean(r_win))
            feat[f"vol_{lbl}"]  = float(np.std(r_win) if len(r_win) > 1 else 0.0)

        # EWMA volatility (lambda = 0.94)
        lam = self._EWMA_LAMBDA
        w_exp = np.array([(1 - lam) * lam ** i for i in range(n)])[::-1]
        w_exp /= w_exp.sum()
        feat["ewma_vol"] = float(np.sqrt(max(np.dot(w_exp, returns ** 2), 0.0)))

        # Price momentum at 3 lookbacks
        for w, lbl in [(6, "30m"), (24, "2h"), (288, "24h")]:
            w = min(w, len(prices) - 1)
            ref = float(prices[-w - 1]) if w > 0 else p
            feat[f"mom_{lbl}"] = float((p - ref) / ref) if ref != 0 else 0.0

        # Volatility regime (recent / long-run)
        rv = float(np.std(returns[-min(12, n):]))
        lv = float(np.std(returns))
        feat["vol_regime"] = rv / (lv + 1e-12)

        # Last return (AR feature)
        feat["ret_last"] = float(returns[-1])

        # Cyclical time-of-day / day-of-week features
        if times is not None and len(times) > 0:
            try:
                dt = datetime.fromtimestamp(float(times[-1]), tz=timezone.utc)
                feat["hour_sin"] = float(np.sin(2 * np.pi * dt.hour / 24))
                feat["hour_cos"] = float(np.cos(2 * np.pi * dt.hour / 24))
                feat["dow_sin"]  = float(np.sin(2 * np.pi * dt.weekday() / 7))
                feat["dow_cos"]  = float(np.cos(2 * np.pi * dt.weekday() / 7))
            except Exception:
                feat["hour_sin"] = feat["hour_cos"] = feat["dow_sin"] = feat["dow_cos"] = 0.0
        else:
            feat["hour_sin"] = feat["hour_cos"] = feat["dow_sin"] = feat["dow_cos"] = 0.0

        return feat

    def _build_dataset(self, prices, returns, times, min_window=48):
        """Rolling-window dataset for NGBoost: X = features at t, y = return at t+1."""
        n = len(returns)
        if n < min_window + 1:
            return None, None
        rows_X, rows_y = [], []
        t_arr = np.array(times) if times is not None else None
        for i in range(min_window, n):
            f = self._features(prices[: i + 1], returns[:i], t_arr[:i] if t_arr is not None else None)
            rows_X.append(f)
            rows_y.append(returns[i])
        if not rows_X:
            return None, None
        if self._fcols is None:
            self._fcols = list(rows_X[0].keys())
        X = np.array([[r[c] for c in self._fcols] for r in rows_X], dtype=np.float64)
        y = np.array(rows_y, dtype=np.float64)
        mask = np.isfinite(y)
        X = np.where(np.isfinite(X), X, 0.0)
        return X[mask], y[mask]

    def _train_ngboost(self, prices, returns, times):
        """Train NGBoost with CRPScore on historical rolling windows."""
        try:
            from ngboost import NGBRegressor
            from ngboost.distns import Normal
            from ngboost.scores import CRPScore
        except ImportError:
            return None

        X, y = self._build_dataset(prices, returns, times)
        if X is None or len(y) < 30:
            return None
        try:
            model = NGBRegressor(
                Dist=Normal,
                Score=CRPScore,
                n_estimators=self._NGBOOST_TREES,
                learning_rate=0.05,
                minibatch_frac=0.8,
                verbose=False,
                random_state=42,
            )
            model.fit(X, y)
            return model
        except Exception:
            return None

    # =========================================================================
    # Layer 4 – Quantile calibration
    # =========================================================================

    def _quantile_scale(self, returns: np.ndarray, df: float) -> float:
        """
        Empirical / theoretical interquantile ratio for scale calibration.
        Compares the empirical (5th, 95th) spread to the Student-t(df)
        theoretical spread, giving a multiplicative correction to sigma.
        """
        if len(returns) < 50:
            return 1.0
        centered = returns - np.median(returns)
        emp_iqr = float(np.percentile(centered, 95) - np.percentile(centered, 5))
        if emp_iqr <= 0:
            return 1.0
        t_iqr = float(sp_stats.t.ppf(0.95, df=df) - sp_stats.t.ppf(0.05, df=df))
        if t_iqr <= 0:
            return 1.0
        # implied_sigma = emp_iqr / t_iqr; clip to prevent runaway
        return float(np.clip(emp_iqr / t_iqr, 0.3, 3.0))

    # =========================================================================
    # Main predict
    # =========================================================================

    def predict(self, asset: str, horizon: int, step: int):
        res = self._BASE_RES
        pts = self.prices.get_prices(asset, days=self._HISTORY_DAYS, resolution=res)
        if not pts or len(pts) < 20:
            return []

        times_all, prices_all = zip(*pts)
        prices_all = np.array(prices_all, dtype=np.float64)
        times_all  = np.array(times_all,  dtype=np.float64)
        returns     = np.diff(prices_all)
        ret_times   = times_all[1:]

        if len(returns) < 10:
            return []

        n_calls = self._calls.get(asset, 0)
        refresh  = (n_calls % 20 == 0)

        # ── Layer 1: degrees of freedom ────────────────────────────────────
        if asset not in self._df or refresh:
            self._df[asset] = self._estimate_df(returns)
        df = self._df[asset]

        # ── Layer 2: GARCH fit ─────────────────────────────────────────────
        if asset not in self._garch or refresh:
            self._garch[asset] = self._fit_garch(returns)
        omega, alpha, beta, sigma2_now = self._garch[asset]

        # ── Layer 3: NGBoost ───────────────────────────────────────────────
        if asset not in self._ngb or n_calls % self._RETRAIN_EVERY == 0:
            mdl = self._train_ngboost(prices_all, returns, ret_times)
            if mdl is not None:
                self._ngb[asset] = mdl

        ngb = self._ngb.get(asset, None)

        ngb_feat_vec = None
        if ngb is not None:
            try:
                cf = self._features(prices_all, returns, ret_times)
                ngb_feat_vec = np.array(
                    [[cf[c] for c in self._fcols]], dtype=np.float64
                )
                ngb_feat_vec = np.where(np.isfinite(ngb_feat_vec), ngb_feat_vec, 0.0)
            except Exception:
                ngb_feat_vec = None

        # ── Layer 4: quantile calibration ──────────────────────────────────
        if asset not in self._scale or refresh:
            self._scale[asset] = self._quantile_scale(returns, df)
        cal = self._scale[asset]

        # EWMA-weighted baseline mean
        lam   = self._EWMA_LAMBDA
        n_r   = len(returns)
        w_exp = np.array([(1 - lam) * lam ** i for i in range(n_r)])[::-1]
        w_exp /= w_exp.sum()
        mu_base  = float(np.dot(w_exp, returns))
        step_sc  = step / res         # scale factor: step size vs 5-min resolution
        num_segs = horizon // step

        self._calls[asset] = n_calls + 1

        distributions = []
        for k in range(1, num_segs + 1):
            # ── Layer 2: GARCH h-step variance ─────────────────────────────
            s2_k    = self._garch_forecast(omega, alpha, beta, sigma2_now, k)
            sig_k   = float(np.sqrt(s2_k)) * np.sqrt(step_sc) * cal

            # ── Mean: NGBoost if available, else EWMA baseline ─────────────
            mu_k = mu_base * step_sc
            if ngb_feat_vec is not None:
                try:
                    dpred   = ngb.pred_dist(ngb_feat_vec)
                    mu_ngb  = float(dpred.params["loc"][0])   * step_sc
                    sig_ngb = float(dpred.params["scale"][0]) * np.sqrt(step_sc) * cal
                    mu_k    = 0.5 * mu_k  + 0.5 * mu_ngb
                    sig_k   = 0.5 * sig_k + 0.5 * sig_ngb
                except Exception:
                    pass

            sig_k = max(sig_k, abs(mu_k) * 0.1 + 1e-6)

            # ── Layer 1: two-component Student-t mixture ───────────────────
            #  • Main (75 %): Student-t(df)     – captures typical fat-tail regime
            #  • Jump (25 %): Student-t(df-1, 2×scale) – extreme events / jumps
            df_jump = float(max(df - 1.0, 2.5))
            components = [
                {
                    "density": {
                        "type": "builtin", "name": "t",
                        "params": {"df": df, "loc": mu_k, "scale": sig_k},
                    },
                    "weight": 0.75,
                },
                {
                    "density": {
                        "type": "builtin", "name": "t",
                        "params": {"df": df_jump, "loc": mu_k, "scale": sig_k * 2.0},
                    },
                    "weight": 0.25,
                },
            ]

            distributions.append({
                "step": k * step,
                "type": "mixture",
                "components": components,
            })

        return distributions


## Configuration

In [ ]:
# ── Assets ──────────────────────────────────────────────────────────────────
assets = ["BTC", "ETH", "SOL", "XAUT", "SPYX", "NVDAX", "TSLAX", "AAPLX", "GOOGLX"]
print("Supported assets:", ", ".join(SUPPORTED_ASSETS))
print("Selected assets :", ", ".join(assets))

# ── Forecast profile ────────────────────────────────────────────────────────
ACTIVE_HORIZON = "24h"   # options: "24h", "1h"

HORIZON  = FORECAST_PROFILES[ACTIVE_HORIZON]["horizon"]
STEPS    = FORECAST_PROFILES[ACTIVE_HORIZON]["steps"]
INTERVAL = FORECAST_PROFILES[ACTIVE_HORIZON]["interval"]
print(f"\nHorizon: {ACTIVE_HORIZON}  |  steps: {STEPS}  |  interval: {INTERVAL}s")

# ── Time window ─────────────────────────────────────────────────────────────
evaluation_end = datetime.now(timezone.utc)
days           = 5    # days of test data
days_history   = 30   # warm-up history

# ── Output directory ─────────────────────────────────────────────────────────
base_dir_results = "results_advanced"
os.makedirs(base_dir_results, exist_ok=True)


## Load Data

In [ ]:
# Load test prices
test_asset_prices = load_test_prices_once(assets, evaluation_end, days=days)

# Load warm-up history (tracker sees this before any evaluation tick)
initial_histories = load_initial_price_histories_once(
    assets, evaluation_end,
    days_history=days_history, days_offset=days
)

visualize_price_data(
    history_data=initial_histories, test_data=test_asset_prices,
    selected_assets=None, show_graph=True
)


## Run Evaluation

In [ ]:
tracker          = AdvancedProbabilisticTracker()
tracker_evaluator = TrackerEvaluator(tracker)

for asset, history_test_prices in test_asset_prices.items():

    # First tick: give the tracker the full warm-up history
    tracker_evaluator.tick({asset: initial_histories[asset]})

    prev_ts        = 0
    predict_count  = 0
    predictions_evaluated = []
    pbar = tqdm(
        desc=f"Evaluating {asset}",
        total=count_evaluations(history_test_prices, HORIZON, INTERVAL),
        unit="eval",
    )

    for ts, price in history_test_prices:
        tracker_evaluator.tick({asset: [(ts, price)]})

        if ts - prev_ts >= INTERVAL:
            prev_ts = ts
            predictions_evaluated = tracker_evaluator.predict(asset, HORIZON, STEPS)

            if predictions_evaluated:
                pbar.update(1)

            if predictions_evaluated and predict_count % 20 == 0:
                pbar.write(
                    f"[{asset}] avg CRPS={tracker_evaluator.overall_score_asset(asset):.4f}"
                    f"  recent={tracker_evaluator.recent_score_asset(asset):.4f}"
                )
            predict_count += 1

    pbar.write(
        f"[{asset}] FINAL  avg CRPS={tracker_evaluator.overall_score_asset(asset):.4f}"
        f"  recent={tracker_evaluator.recent_score_asset(asset):.4f}"
    )
    pbar.close()
    print()

tracker_name = tracker_evaluator.tracker.__class__.__name__
print(f"\nTracker: {tracker_name}")
print(f"Overall normalised CRPS: {tracker_evaluator.overall_score():.4f}")

current_results_dir = tracker_evaluator.to_json(
    horizon=HORIZON, steps=STEPS, interval=INTERVAL,
    base_dir=base_dir_results
)


## Visualise Results

In [ ]:
# Timeline of scores
timestamped_scores = tracker_evaluator.scores
print("\n(Note – a score at time t evaluates a forecast issued at t − horizon)")

if timestamped_scores:
    plot_scores(timestamped_scores)
else:
    print("No scored predictions yet.")


In [ ]:
# Density forecast plot for the last prediction
if predictions_evaluated:
    plot_quarantine(
        asset, predictions_evaluated[0], step=STEPS[0],
        prices=tracker_evaluator.price_provider,
    )


## Submit

1. **Download** this notebook
2. **Upload** to the CrunchDAO platform
3. **Create a run** to validate

👉 https://hub.crunchdao.com/competitions/synth/submit/notebook

*Ensure predictions complete within 40 seconds per asset.*
